In [0]:
from pyspark.sql import functions as F

lbtt2 = spark.table("scottish_housing_ws.silver.lbtt_table_2")

In [0]:
lbtt2.printSchema()
lbtt2.show(3)

In [0]:
gold = (
    lbtt2
    .withColumn(
        "band_up_to_145k_pct",
        F.round(F.col("band_up_to_145k_returns") / F.col("total_returns") * 100, 1)
    )
    .withColumn(
        "band_145k_to_250k_pct",
        F.round(F.col("band_145k_to_250k_returns") / F.col("total_returns") * 100, 1)
    )
    .withColumn(
        "band_250k_to_325k_pct",
        F.round(F.col("band_250k_to_325k_returns") / F.col("total_returns") * 100, 1)
    )
    .withColumn(
        "band_325k_to_750k_pct",
        F.round(F.col("band_325k_to_750k_returns") / F.col("total_returns") * 100, 1)
    )
    .withColumn(
        "band_over_750k_pct",
        F.round(F.col("band_over_750k_returns") / F.col("total_returns") * 100, 1)
    )
    .select(
        "report_date",
        "year_month",
        "is_provisional",
        "band_up_to_145k_returns",
        "band_up_to_145k_pct",
        "band_145k_to_250k_returns",
        "band_145k_to_250k_pct",
        "band_250k_to_325k_returns",
        "band_250k_to_325k_pct",
        "band_325k_to_750k_returns",
        "band_325k_to_750k_pct",
        "band_over_750k_returns",
        "band_over_750k_pct",
        "total_returns",
        "total_tax_ex_ads"
    )
    .orderBy("report_date")
)

In [0]:

print(f"Row count: {gold.count()}")
gold.orderBy("report_date").show(3)
gold.orderBy(F.col("report_date").desc()).show(3)

# Quick sanity check -- band percentages should sum to ~100
gold.withColumn(
    "pct_sum",
    F.col("band_up_to_145k_pct") +
    F.col("band_145k_to_250k_pct") +
    F.col("band_250k_to_325k_pct") +
    F.col("band_325k_to_750k_pct") +
    F.col("band_over_750k_pct")
).select("year_month", "pct_sum").orderBy("report_date").show(5)

In [0]:

(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.lbtt_price_band_distribution")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_lbtt_price_band_distribution/")
)

print("Gold 08 complete.")

In [0]:
# Calculate band shares 
# Express each band as a percentage of total returns that month
# This normalises for overall market volume changes over time
# and shows how the mix of transactions is shifting between price bands